In [1]:
import duckdb
con = duckdb.connect()
P = r"C:\Users\ikhairat\DS-GIS-senior\data\raw"   # adjust the path if the notebook runs from m00_environment/notebooks

# 1. Top categories
# Find all Parquet files under the raw-data directory
places = f"read_parquet('{P}\\**\\*.parquet', union_by_name=true)"

con.sql(
    f"""SELECT basic_category, count(*) AS n
        FROM {places}
        GROUP BY 1
        ORDER BY n DESC
        LIMIT 30"""
).show()

# 2. Everything restaurant-like, two ways
con.sql(f"""SELECT taxonomy."primary" AS cat, count(*) n 
            FROM {places}
            WHERE taxonomy."primary" ILIKE '%restaurant%' 
            GROUP BY 1 ORDER BY n DESC"""
).show()

con.sql(f"""SELECT basic_category, count(*) n 
            FROM {places}
            WHERE list_contains(taxonomy.hierarchy, 'restaurant') 
            GROUP BY 1 
            ORDER BY n DESC"""
).show()

# 3. Confidence and operating status
con.sql(f"SELECT round(confidence, 1) c, count(*) n FROM {places} GROUP BY 1 ORDER BY 1").show()
con.sql(f"SELECT operating_status, count(*) n FROM {places} GROUP BY 1").show()

┌────────────────────────────────┬───────┐
│         basic_category         │   n   │
│            varchar             │ int64 │
├────────────────────────────────┼───────┤
│ NULL                           │  2143 │
│ restaurant                     │  1663 │
│ fashion_and_apparel_store      │  1431 │
│ personal_or_beauty_service     │  1084 │
│ professional_service           │  1039 │
│ shopping                       │   987 │
│ hardware_home_and_garden_store │   799 │
│ wellness_service               │   598 │
│ real_estate_service            │   507 │
│ cafe                           │   483 │
│  ·                             │    ·  │
│  ·                             │    ·  │
│  ·                             │    ·  │
│ financial_service              │   303 │
│ coffee_shop                    │   302 │
│ corporate_or_business_office   │   289 │
│ dental_clinic                  │   286 │
│ specialty_school               │   268 │
│ hospital                       │   257 │
│ sport_or_

In [2]:
con.sql("INSTALL spatial; LOAD spatial;")   # ST_X / ST_Y need the spatial extension

flat = con.sql(f"""
    SELECT id,
           names."primary" AS name,
           taxonomy."primary" AS category,
           coalesce(list_contains(taxonomy.hierarchy, 'restaurant'), false) AS is_restaurant,
           confidence,
           ST_X(geometry) AS lon,
           ST_Y(geometry) AS lat
    FROM {places}
    WHERE ST_GeometryType(geometry) = 'POINT'
""").df()

print(flat.shape, flat.is_restaurant.sum(), flat.id.is_unique)
flat.to_parquet(r"C:\Users\ikhairat\DS-GIS-senior\data\processed\places_flat.parquet")  # create data\processed first

(24312, 7) 1663 True
